# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

Hernquist–Ostriker (1992) 基底を G=1、基底長さ尺度a_basis=1の無次元系で用います。入力密度が質量規格化を含むため、再構成ポテンシャルにさらにM/Rを掛けません。係数積分は有限半径・有限格子による近似です。n_max、角次数、積分半径、格子解像度を別々に変えて収束を確認してください。Hernquistの中心密度は発散するため積分格子は r>0 とします。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y, gegenbauer, gamma
from scipy.integrate import simpson
import math

# Plot configuration parameters
parameters = {
    'axes.labelsize': 26,  # Set font size of x and y axis labels to 26
    'xtick.labelsize': 24,  # Set font size of x-axis ticks to 24
    'ytick.labelsize': 24,  # Set font size of y-axis ticks to 24
    'axes.titlesize': 24,   # Set font size of individual axis titles to 24
    'figure.titlesize': 24, # Set font size of the overall figure title to 24
    'legend.fontsize': 18,  # Set font size of the legend to 18
    'figure.figsize': [10, 6],  # Set default figure size to 10 inches by 6 inches
    'axes.linewidth': 2.0  # Set the line width of the axis borders to 2.0
}

# Apply the parameters to matplotlib's default settings
plt.rcParams.update(parameters)

# Examples of Target Density Distributions

In [ ]:
# Sphere parameters
Rsph = 1.0  # Radius
rho0 = 1.0  # Density constant

# Conversion function from spherical to Cartesian coordinates (fixed radius r)
def spherical_to_cartesian(r, theta, phi):
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    return x, y, z

# 1. Uniform spherical distribution (constant density inside radius R, zero outside)
def uniform_sphere(r, theta, phi, R=1.0):
    return np.where(r <= R, rho0, 0.0) * np.ones_like(theta)

# Potential calculation function for a uniformly dense sphere
def potential_uniform_sphere(r, theta, phi, R=1.0):
    M = (4/3) * np.pi * R**3 * rho0  # Mass of the sphere
    Phi_outside = -G * M / r
    Phi_inside = (2 * np.pi * G * rho0 / 3) * r**2 - 2 * np.pi * G * rho0 * R**2
    return np.where(r < R, Phi_inside, Phi_outside)

# 2. Plummer sphere density distribution
def plummer_sphere(r, theta, phi, M=1.0, b=1.0):
    return (3 * M) / (4 * np.pi * b**3) * (1 + (r / b)**2)**(-5/2)

def potential_plummer_sphere(r, theta, phi, M=1.0, b=1.0):
    return -G * M / np.sqrt(r**2 + b**2)

# 3. Hernquist density distribution
def hernquist_sphere(r, theta, phi, M=1.0, a=1.0):
    return (M * a) / (2 * np.pi * r * (r + a)**3)

def potential_hernquist_sphere(r, theta, phi, M=1.0, a=1.0):
    return -G * M / (r + a)

## Hernquist & Ostriker (1992) Biorthogonal Basis Function Set

### Objective:
The goal of this code is to compute the gravitational potential \( \Phi(r, \theta, \phi) \) for a given density distribution \( \rho(r, \theta, \phi) \) using spherical harmonic expansion and radial basis functions. This method involves expanding both the density and the potential in terms of spherical harmonics and radial functions.

### Key Components:

1. **`Knl(n, l)` Function**:
   - This function calculates the eigenvalue \( K_{nl} \), which is a characteristic value used in the radial basis functions for both the density and the potential.
   - **Parameters**:
     - `n`: Radial expansion index.
     - `l`: Angular expansion index.

2. **`rho_basis(r, n, l)` Function**:
   - This function computes the density basis function \( p_{nlm}(r) \), which describes the spatial distribution of mass or density.
   - The radial coordinate \( r \) is transformed using \( x = \frac{r - 1}{r + 1} \) to simplify calculations.
   - **Parameters**:
     - `r`: Radial distance.
     - `n`: Radial expansion index.
     - `l`: Angular expansion index.

3. **`phi_basis(r, n, l)` Function**:
   - This function computes the potential basis function \( \Phi_{nlm}(r) \), which is used in the expansion of the gravitational potential.
   - Similar to `rho_basis`, it uses a variable transformation \( x = \frac{r - 1}{r + 1} \) and Gegenbauer polynomials.
   - **Parameters**:
     - `r`: Radial distance.
     - `n`: Radial expansion index.
     - `l`: Angular expansion index.

4. **`compute_I_nl(n, l)` Function**:
   - This function calculates the normalization constant \( I_{nl} \), which is required to normalize the basis functions.
   - It involves the gamma function \( \Gamma \) and factorials, ensuring that the expansion coefficients are properly scaled.
   - **Parameters**:
     - `n`: Radial expansion index.
     - `l`: Angular expansion index.
   - **Returns**: \( I_{nl} \), the normalization constant.

5. **`compute_potential_for_density` Function**:
   - This is the main function that computes the gravitational potential \( \Phi(r, \theta, \phi) \) using the density distribution \( \rho(r, \theta, \phi) \).
   - **Parameters**:
     - `density_func`: Function that computes the density distribution \( \rho(r, \theta, \phi) \).
     - `r_values`, `theta_values`, `phi_values`: Arrays of radial, polar, and azimuthal coordinates, respectively.
     - `n_max`, `l_max`, `m_max`: Maximum values for the expansion indices \( n \), \( l \), and \( m \).
   - **Process**:
     1. **Mesh Grid Creation**: A mesh grid is created for \( \theta \) and \( \phi \), allowing for calculations over a spherical coordinate system.
     2. **Density Calculation**: The density distribution \( \rho(r, \theta, \phi) \) is calculated over the grid.
     3. **Expansion Coefficients**: The expansion coefficients \( c_{nlm} \) are calculated by integrating the product of the density function, potential basis functions, and spherical harmonics. The integration is performed over \( r \), \( \theta \), and \( \phi \) using Simpson's rule (`simpson`).
     4. **Potential Calculation**: The potential is then reconstructed by summing over the spherical harmonics and radial basis functions, using the computed expansion coefficients.
   - **Returns**: The gravitational potential \( \Phi(r, \theta, \phi) \) as a 3D array of values at the given grid points.

### Key Features:

1. **Gravitational Potential Calculation**:
   - The potential is calculated by expanding both the density and the potential in spherical harmonics and radial basis functions, making it suitable for complex density distributions.

2. **Numerical Integration**:
   - The code uses Simpson's rule to perform the necessary integrals over \( r \), \( \theta \), and \( \phi \), with convergence checks required for the grid and integration boundary.

3. **Normalization**:
   - The function `compute_I_nl` ensures that the expansion functions are properly normalized, making the potential and density consistent.

4. **Modular Design**:
   - Each component of the calculation is separated into its own function, making the code modular and flexible. This allows for easy modification and extension.

### Key Takeaways:
- **Efficient Potential Calculation**: The use of spherical harmonics and radial basis functions allows the code to compute the gravitational potential efficiently for arbitrary density distributions.
- **Numerical Accuracy**: By using Simpson's rule for numerical integration and properly normalizing the basis functions, accuracy must be verified by refining the grid, domain and expansion orders.
- **Flexibility**: The code is flexible in terms of the grid resolution and the number of expansion terms used, allowing it to handle various density profiles and geometries.

In [ ]:
# Constants
G = 1  # Set the gravitational constant to 1 (can be scaled as needed)

def Knl(n, l):
    return 0.5 * n * (n + 4*l + 3) + (l + 1) * (2*l + 1)

def rho_basis(r, n, l):
    """Density basis function p_nlm"""
    alpha = 2 * l + 1.5  # Parameter for Gegenbauer polynomials
    x = (r - 1) / (r + 1)  # Variable transformation
    return (0.5*Knl(n,l)/np.pi) * np.sqrt(4.*np.pi) * (r**(l-1) / (1 + r)**(2*l + 3)) * gegenbauer(n, alpha)(x)

def phi_basis(r, n, l):
    """Potential basis function Φ_nlm"""
    alpha = 2 * l + 1.5  # Parameter for Gegenbauer polynomials
    x = (r - 1) / (r + 1)  # Variable transformation
    return - np.sqrt(4.*np.pi) * (r**l / (1 + r)**(2*l + 1)) * gegenbauer(n, alpha)(x)

def compute_I_nl(n, l):
    """
    Calculate normalization constant I_nl
    Parameters:
        n (int): Basis function n
        l (int): Basis function l
    Returns:
        I_nl (float): Normalization constant
    """
    numerator = -Knl(n, l) * (4 * np.pi) * gamma(n + 4*l + 3)
    denominator = 2**(8*l + 6) * math.factorial(n) * (n + 2*l + 3/2) * (gamma(2*l + 1.5)**2)
    I_nl = numerator / denominator
    return I_nl

# Function to calculate the potential based on the selected density distribution
def compute_potential_for_density(density_func, r_values, theta_values, phi_values, n_max=5, l_max=5, m_max=None):
    """
    Parameters:
        density_func: Function to calculate the density distribution
        r_values: Array of radial positions
        theta_values: Array of θ values
        phi_values: Array of φ values
        n_max: Maximum value for n in the basis functions
        l_max: Maximum value for l in the basis functions
        m_max: Maximum value for m in the basis functions (default is l_max)
    Returns:
        potential: 3D array of potential values (r, theta, phi)
    """
    if m_max is None:
        m_max = l_max  # Full azimuthal range
    if not (0 <= m_max <= l_max):
        raise ValueError("Require 0 <= m_max <= l_max")
    
    # Create mesh grid
    theta, phi = np.meshgrid(theta_values, phi_values, indexing='ij')  # theta: (theta, phi), phi: (theta, phi)
    
    # Calculate density based on the provided density function
    rho_values = density_func(r_values[:, None, None], theta, phi)  # r: (r, theta, phi)
    
    # Calculate expansion coefficients
    coefficients = np.zeros((n_max+1, l_max+1, 2*m_max+1), dtype=complex)  # Use complex type
    for n in range(n_max+1):
        for l in range(l_max+1):
            for m in range(-min(l, m_max), min(l, m_max)+1):
                # Calculate the basis functions (using potential basis function)
                phi_nl = phi_basis(r_values, n, l)[:, None, None]  # (r, 1, 1)
                Y_lm   = sph_harm_y(l, m, theta, phi)  # (theta, phi)

                # Function to integrate
                integrand = rho_values * phi_nl * np.conj(Y_lm) * r_values[:, None, None]**2 * np.sin(theta)
                # 1. Integrate over φ
                integral_phi   = simpson(integrand, x=phi_values, axis=2)
                # 2. Integrate over θ
                integral_theta = simpson(integral_phi, x=theta_values, axis=1)
                # 3. Integrate over r
                coefficient    = simpson(integral_theta, x=r_values, axis=0)
                
                # Store the expansion coefficients
                coefficients[n, l, m + m_max] = coefficient / compute_I_nl(n, l)

                # Output the expansion coefficients and values of n, l, m
                # print(f"n={n}, l={l}, m={m}, coefficient={np.real(coefficients[n, l, m + m_max]),np.imag(coefficients[n, l, m + m_max])}")
    
    # Calculate potential (3D array)
    potential = np.zeros((len(r_values), len(theta_values), len(phi_values)), dtype=complex)
    for n in range(n_max+1):
        for l in range(l_max+1):
            for m in range(-min(l, m_max), min(l, m_max)+1):
                # Calculate the basis functions
                phi_nl = phi_basis(r_values, n, l)[:, None, None]  # (r, 1, 1)
                Y_lm   = sph_harm_y(l, m, theta, phi)  # (theta, phi)
                # Accumulate the potential
                potential += coefficients[n, l, m + m_max] * phi_nl * Y_lm
    
    # Take the real part to ensure the potential is real
    potential = np.real(potential)
    return potential  # (r, theta, phi)

### Objective:
The code aims to visualize the potential basis functions \( \Phi_{nl}(r) \) for different values of \( n \) and \( l \). The basis functions are computed over a specified range of radial values \( r \) and plotted to observe their behavior.

In [ ]:
# Set the range for r
r_values = np.linspace(0.01, 10, 500)  # 500 points from r = 0.01 to 10

# Calculate and plot potential basis functions for n = 0, 1, 2, 3
n_values = [0, 1, 2, 3]
l_values = [0, 1, 2, 3, 4, 5]

# Set the colormap to 'Greens'
cmap = plt.get_cmap('Greens')

# Set up the subplots
size = 6
fig, axs = plt.subplots(1, len(n_values), figsize=(4 * size, size))  # 1 row, 4 columns

# Plot the potential basis functions for each n and l = [0, 1, 2, 3, 4, 5]
for j, n in enumerate(n_values):
    for l in l_values:
        phi_nl_values = phi_basis(r_values, n, l)
        # Get color from the colormap based on l
        color = cmap(1.0 - 0.9 * l / len(l_values))  # Green color based on the value of l
        axs[j].plot(r_values, phi_nl_values, label=rf"$\Phi_{{{n}{l}}}(r)$", lw=3, color=color)
    
    axs[j].set_xlabel('r')
    axs[j].set_ylabel(rf'$\Phi_{{{n}l}}(r)$')
    axs[j].set_title(rf'$n={n}$')
    axs[j].set_ylim(-5, 5)
    
    # Set grid ticks at intervals of 5
    axs[j].set_xticks(np.arange(0, 11, 2))
    axs[j].set_yticks(np.arange(-5, 6, 2))
    axs[j].grid(True, which='both', linestyle='--')
    axs[j].legend(loc="lower right")

# Display the graph
plt.tight_layout()
plt.show()

### Objective:
The code visualizes the potential basis functions \( \Phi_{nl}(r) \) for different expansion indices \( n \) and \( l \) over a range of radial values \( r \). The goal is to understand how the potential functions change with varying \( n \) and \( l \) values.

In [ ]:
# Set the range for r
r_values = np.linspace(0.01, 10, 500)  # 500 points from r = 0.01 to 10

# Calculate and plot potential basis functions for n = 0, 1, 2, 3, 4, 5
n_values = [0, 1, 2, 3, 4, 5]
l_values = [0, 1, 2]

# Get the colormap
cmap = plt.get_cmap('Reds')

# Set up the subplots: arrange in a horizontal row (1 row, 4 columns)
size = 6
fig, axs = plt.subplots(1, len(l_values), figsize=(size * len(l_values), size))  # 1 row, 4 columns

# Plot the potential basis functions for n = [0, 1, 2, 3] for each l
for i, l in enumerate(l_values):
    for n in n_values:
        phi_nl_values = phi_basis(r_values, n, l)
        # Get color corresponding to n from the colormap
        color = cmap(1.0 - 0.9 * n / len(n_values))  # Reds color based on n value
        axs[i].plot(r_values, phi_nl_values, label=rf"$\Phi_{{{n}{l}}}(r)$", lw=3, color=color)
    
    axs[i].set_xlabel('r')
    axs[i].set_ylabel(rf'$\Phi_{{n{l}}}(r)$')
    axs[i].set_title(rf'$l={l}$ for $n=[0, 1, 2, 3]$')
    axs[i].set_ylim(-10, 10)
    
    # Set grid ticks at intervals of 5
    axs[i].set_xticks(np.arange(0, 11, 2))
    axs[i].set_yticks(np.arange(-10, 11, 5))
    axs[i].grid(True, which='both', linestyle='--')
    axs[i].legend(loc="lower right")

# Display the graph
plt.tight_layout()
plt.show()

### Objective:
This code compares the gravitational potential of a Hernquist sphere, a common model used in astrophysics, using two approaches:
1. **Analytic Solution**: The exact gravitational potential for a Hernquist sphere.
2. **Computed Solution**: The gravitational potential derived from the basis set expansion method.

In [ ]:
# Example usage: Select a Hernquist spherical distribution
r_values = np.linspace(0.01, 10, 100)
theta_values = np.linspace(0, np.pi, 100)
phi_values = np.linspace(0, 2 * np.pi, 100)

# Hernquist distribution
M = 1.0
a = 1.0
potential_bse = compute_potential_for_density(hernquist_sphere, r_values, theta_values, phi_values, n_max=0, l_max=0, m_max=None)
potential_ana = potential_hernquist_sphere(r_values, theta_values, phi_values)

# Display the calculation results
plt.plot(r_values, potential_ana, label='Analytic', linestyle='--', color='r', lw=5)
plt.plot(r_values, potential_bse[:, 0, 0], label='Computed', color='b', lw=3)
plt.xlabel('Radius r')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Hernquist Sphere')
plt.grid(True)
plt.legend()
plt.show()

### Objective:
This code calculates and compares the gravitational potential of a **uniform spherical mass distribution** using two approaches:
1. **Analytic Solution**: The exact gravitational potential for a uniform spherical distribution.
2. **Computed Solution**: The gravitational potential derived from a basis set expansion method.

In [ ]:
# Example usage: Select a uniform spherical distribution
r_values = np.linspace(0.01, 10, 100)
theta_values = np.linspace(0, np.pi, 100)
phi_values = np.linspace(0, 2 * np.pi, 100)

# Choose the density function for the uniform spherical distribution
R = 1.0
M = (4/3) * np.pi * R**3 * rho0  # Density already includes this mass
potential_uniform = compute_potential_for_density(uniform_sphere, r_values, theta_values, phi_values, n_max=5, l_max=0, m_max=None)
potential_uniform_analytic = potential_uniform_sphere(r_values, theta_values, phi_values)

# Display the calculation results
plt.plot(r_values, potential_uniform_analytic, label='Analytic', linestyle='--', color='r', lw=5)
plt.plot(r_values, potential_uniform[:, 0, 0], label='Computed', color='b', lw=3)
plt.xlabel('Radius r')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Uniform Sphere')
plt.grid(True)
plt.legend()
plt.show()

### Objective:
This code calculates and compares the **gravitational potential** of a **uniform spherical mass distribution** using two approaches:
1. **Analytic Solution**: The exact gravitational potential for a uniform spherical distribution.
2. **Computed Solution**: The gravitational potential derived using a basis set expansion (BSE) method, varying the maximum order of expansion $n_{\text{max}}$.

In [ ]:
# Example usage: Select a uniform spherical distribution
r_values = np.linspace(0.01, 10, 100)
theta_values = np.linspace(0, np.pi, 100)
phi_values = np.linspace(0, 2 * np.pi, 100)

# Choose the density function for the uniform spherical distribution
R = 1.0
M = (4/3) * np.pi * R**3 * rho0  # Density already includes this mass
potential_uniform_analytic = potential_uniform_sphere(r_values, theta_values, phi_values)

# Plot the analytic potential
plt.plot(r_values, potential_uniform_analytic, label='Analytic', linestyle='--', color='r', lw=5)

# Compute and plot the potential for different n_max values
for n_max in range(1, 11, 2):  # Vary n_max from 1 to 10 in steps of 2
    potential_uniform = compute_potential_for_density(uniform_sphere, r_values, theta_values, phi_values, n_max, l_max=0, m_max=None)
    plt.plot(r_values, potential_uniform[:, 0, 0], label='Computed w/ $n_{max}=$' + str(n_max), lw=3)

# Configure plot labels and title
plt.xlabel('Radius $r$')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Uniform Sphere')
plt.xlim(0, 3)
plt.grid(True)
plt.legend()
plt.show()

### Objective:
This code calculates and compares the **gravitational potential** of a **Plummer spherical distribution** using two approaches:
1. **Analytic Solution**: The exact gravitational potential for a Plummer sphere.
2. **Computed Solution**: The gravitational potential derived using a basis set expansion (BSE) method.

In [ ]:
# Example usage: Select a Plummer spherical distribution
r_values = np.linspace(0.01, 10, 100)
theta_values = np.linspace(0, np.pi, 100)
phi_values = np.linspace(0, 2 * np.pi, 100)

# Choose the density function for the Plummer sphere
R = 1.0
M = 1.0
potential_uniform = compute_potential_for_density(plummer_sphere, r_values, theta_values, phi_values, n_max=5, l_max=0, m_max=None)
potential_uniform_analytic = potential_plummer_sphere(r_values, theta_values, phi_values)

# Display the calculation results
plt.plot(r_values, potential_uniform_analytic, label='Analytic', linestyle='--', color='r', lw=5)
plt.plot(r_values, potential_uniform[:, 0, 0], label='Computed', color='b', lw=3)
plt.xlabel('Radius r')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Plummer Sphere')
plt.grid(True)
plt.legend()
plt.show()

### Objective:
This code calculates and visualizes the gravitational potential of a **Plummer sphere** using both an analytical solution and a computed solution derived from a basis set expansion (BSE) method. It allows for the comparison of potentials calculated with varying values of $n_{\text{max}}$.

In [ ]:
# Example usage: Select a uniform spherical distribution
r_values = np.linspace(0.01, 10, 100)
theta_values = np.linspace(0, np.pi, 100)
phi_values = np.linspace(0, 2 * np.pi, 100)

# Choose the density function for the Plummer sphere
R = 1.0
M = 1.0

# Calculate the analytical potential for the Plummer sphere
potential_uniform_analytic = potential_plummer_sphere(r_values, theta_values, phi_values)

# Plot the analytical potential
plt.plot(r_values, potential_uniform_analytic, label='Analytic', linestyle='--', color='r', lw=5)

# Compute and plot the potential for different values of n_max
for n_max in range(1, 6, 2):  # n_max values: 1, 3, 5
    potential_uniform = compute_potential_for_density(plummer_sphere, r_values, theta_values, phi_values, n_max, l_max=0, m_max=None)
    plt.plot(r_values, potential_uniform[:, 0, 0], label=f'Computed w/ $n_{{max}}=$ {n_max}', lw=3)

# Set labels, title, and plot limits
plt.xlabel('Radius r')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Plummer Sphere')
plt.xlim(0, 3)
plt.grid(True)
plt.legend()
plt.show()

### Objective:
This code defines a **non-spherical density distribution** for an ellipsoidal distribution and calculates the corresponding gravitational potential. It demonstrates how to handle more complex density shapes beyond spherical symmetry.

In [ ]:
# Example of defining a non-spherical density distribution: Ellipsoidal distribution
def ellipsoidal_sphere(r, theta, phi, a=1.0, b=1.5, c=2.0):
    """Density function for an ellipsoidal distribution"""
    # Ellipsoidal radius
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    ellipsoid = (x/a)**2 + (y/b)**2 + (z/c)**2
    rho0 = 1.0  # Uniform density
    return rho0 * (ellipsoid <= 1.0)

# Example usage: Select an ellipsoidal distribution
r_values = np.linspace(0.1, 5, 50)  # Range and resolution for radius
theta_values = np.linspace(0, np.pi, 31)  # Range and resolution for theta
phi_values = np.linspace(0, 2 * np.pi, 61)  # Range and resolution for phi

# Select the density function for the ellipsoidal distribution
def density_func(r, theta, phi):
    return ellipsoidal_sphere(r, theta, phi, a=1.0, b=1.5, c=2.0)

# Calculate the gravitational potential
potential_ellipsoidal = compute_potential_for_density(
    density_func, r_values, theta_values, phi_values, n_max=10, l_max=5, m_max=None
)

# Potential along the x-axis (θ = π/2, φ = 0)
theta_x = np.pi / 2
phi_x = 0
theta_x_index = np.argmin(np.abs(theta_values - theta_x))
phi_x_index = np.argmin(np.abs(phi_values - phi_x))
potential_x = potential_ellipsoidal[:, theta_x_index, phi_x_index]

# Potential along the y-axis (θ = π/2, φ = π/2)
theta_y = np.pi / 2
phi_y = np.pi / 2
theta_y_index = np.argmin(np.abs(theta_values - theta_y))
phi_y_index = np.argmin(np.abs(phi_values - phi_y))
potential_y = potential_ellipsoidal[:, theta_y_index, phi_y_index]

# Potential along the z-axis (θ = 0)
theta_z = 0
phi_z = 0  # The value of φ does not matter
theta_z_index = np.argmin(np.abs(theta_values - theta_z))
phi_z_index = 0  # Independent of φ value
potential_z = potential_ellipsoidal[:, theta_z_index, phi_z_index]

# Plot the results
plt.figure(figsize=(10, 6))

# Potential along the x-axis
plt.plot(r_values, potential_x, label='x-axis', lw=3)

# Potential along the y-axis
plt.plot(r_values, potential_y, label='y-axis', lw=3)

# Potential along the z-axis
plt.plot(r_values, potential_z, label='z-axis', lw=3)

# Plot settings
plt.xlabel('Radius r')
plt.ylabel('Potential $\\Phi(r)$')
plt.title('Gravitational Potential for Ellipsoidal Sphere')
plt.grid(True)
plt.legend()
plt.show()